# 02 — Drop in over the OpenAI SDK

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenTracy/opentracy/blob/main/notebooks/02_drop_in_openai.ipynb)

The shortest path to adopting OpenTracy is to **not install a new SDK**. Keep your existing `openai` Python client — just change `base_url`. Your code keeps working, every call is traced, and you can switch providers or point at a distilled student later without touching the app again.

In this notebook:

1. Show that a plain OpenAI SDK call works as-is
2. Start a local OpenTracy engine
3. Point the same OpenAI client at the engine — **one line of change**
4. See that the same code now routes through OpenTracy (and can call any of 13 providers)

> **Runtime:** CPU is fine. You need an OpenAI API key.

## Setup

In [1]:
!pip install opentracy openai -q

In [ ]:
# In an interactive session this prompts; in a non-interactive run
# (e.g. `jupyter nbconvert --execute`) we just skip the prompt and
# let downstream cells handle the missing-key case.
import os, sys
if not os.environ.get("OPENAI_API_KEY") and sys.stdin.isatty():
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")


## 1. The "before" — plain OpenAI SDK

This is the code you already have. It talks straight to OpenAI.

In [ ]:
# 'The before': plain OpenAI SDK going direct to OpenAI. Needs a real
# OPENAI_API_KEY in the environment (the engine's stored key doesn't apply
# here — this call bypasses the engine entirely on purpose).
from openai import OpenAI

try:
    client = OpenAI()   # picks up OPENAI_API_KEY from env
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Hello in three words."}],
    )
    print(resp.choices[0].message.content)
except Exception as e:
    print(f"direct OpenAI call failed ({type(e).__name__}): {e}")
    print("\nThat's OK — the rest of the notebook goes through the engine,")
    print("which has its own provider key. This cell just demonstrates what")
    print("your code looked like BEFORE adopting OpenTracy.")


Works. But you have no cost data, no latency data, no trace, and you're locked into OpenAI at the SDK level. If you wanted to try Anthropic tomorrow, you'd install a new SDK and rewrite this block.

## 2. Start the OpenTracy engine

The engine is a small Go binary bundled inside the `opentracy` wheel. We spawn it in the background — this is usually ~2 seconds in Colab.

In [5]:
# ---- OpenTracy platform hook-up (optional but usually what you want) ----
# By default, opentracy.completion() and opentracy.Router go DIRECT to the
# provider (OpenAI, Anthropic, etc.). That means traces, cost accounting,
# routing decisions, and the distillation loop will NOT see your calls.
#
# To make every call pass through the running OpenTracy engine so it
# shows up in the dashboard at http://<host>:3000/ — uncomment the two
# lines below BEFORE the `import opentracy` call in the next cell.
#
# Provider API keys are configured once via the UI (Settings → API Keys)
# or POST /v1/secrets/<provider>; the engine reads them from
# ~/.opentracy/secrets.json inside its container on every call.

import os
os.environ["OPENTRACY_ENGINE_URL"] = "http://localhost:8080"  # or your remote host


In [ ]:
# Prefer a running OpenTracy engine (from docker-compose up or a remote
# host). Fall back to spinning one up inline, which requires the routing
# weights to be on disk — download them once with
# `opentracy download weights-mmlu-v1` if you don't have them yet.
import os
import httpx

ENGINE_URL = os.environ.get("OPENTRACY_ENGINE_URL", "http://localhost:8080")
engine = None  # set when we spin one up in-process


def _is_engine_up(url: str) -> bool:
    try:
        return httpx.get(f"{url}/health", timeout=1.5).status_code == 200
    except Exception:
        return False


if _is_engine_up(ENGINE_URL):
    print(f"engine already up at {ENGINE_URL} (reusing it)")
else:
    try:
        from opentracy.engine import GoEngine
        engine = GoEngine()
        engine.start()
        ENGINE_URL = engine.url
        print(f"engine up at {ENGINE_URL}")
    except FileNotFoundError as e:
        raise RuntimeError(
            "No running engine at " + ENGINE_URL + " and no routing weights "
            "locally. Point OPENTRACY_ENGINE_URL at a running engine (e.g. the "
            "docker-compose stack on your host) OR run `opentracy download "
            "weights-mmlu-v1` and re-run this cell."
        ) from e

print(f"health: {httpx.get(f'{ENGINE_URL}/health').json()}")


In [ ]:
# Probe what the engine can do RIGHT NOW so the rest of the notebook can
# skip cells that would otherwise 400/500 (auto routing needs weights,
# provider calls need the matching API key to be saved on the engine).
health = httpx.get(f"{ENGINE_URL}/health").json()
keys = httpx.get(f"{ENGINE_URL}/v1/config/keys").json()

ROUTING_AVAILABLE = int(health.get("num_clusters", 0)) > 0
CONFIGURED_PROVIDERS = set(keys.get("configured_providers", []) or [])

print(f"routing weights loaded:  {ROUTING_AVAILABLE}  (num_clusters={health.get('num_clusters')})")
print(f"providers with API keys: {sorted(CONFIGURED_PROVIDERS) or '(none — set one via POST /v1/config/keys or the UI)'}")

if not ROUTING_AVAILABLE:
    print("\nThe model='auto' cell below will be skipped until you load weights:")
    print("  opentracy download weights-mmlu-v1")


## 3. The "after" — same OpenAI SDK, new `base_url`

**This is the only change to your app.** Point the client at the engine URL. `api_key` can be any non-empty string — the engine holds provider keys.

In [ ]:
client = OpenAI(
    base_url=f"{ENGINE_URL}/v1",
    api_key="any",           # engine holds the real provider key
)

resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",    # provider/model — only new convention
    messages=[{"role": "user", "content": "Explain TCP in one sentence."}],
)

print(resp.choices[0].message.content)

Same code. Same response shape. But now every call flowed through OpenTracy — we can inspect what happened.

In [ ]:
# OpenTracy-specific response headers come back on the raw HTTP response.
# The OpenAI SDK swallows headers; peek via the low-level client instead:
raw = client.with_raw_response.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "TCP, one sentence."}],
)
print("selected model:", raw.headers.get("x-opentracy-selected-model"))
print("cluster:       ", raw.headers.get("x-opentracy-cluster-id"))
print("routing ms:    ", raw.headers.get("x-opentracy-routing-ms"))

## 4. Switch providers — still the same SDK

Because the engine speaks all 13 provider protocols, trying a different model is a **string change** — not an SDK migration.

In [ ]:
# Groq (uncomment if you have GROQ_API_KEY set on the engine)
# resp = client.chat.completions.create(
#     model="groq/llama-3.3-70b-versatile",
#     messages=[{"role": "user", "content": "TCP, one sentence."}],
# )
# print(resp.choices[0].message.content)

# Anthropic (uncomment if you have ANTHROPIC_API_KEY set on the engine)
# resp = client.chat.completions.create(
#     model="anthropic/claude-haiku-4-5-20251001",
#     messages=[{"role": "user", "content": "TCP, one sentence."}],
# )
# print(resp.choices[0].message.content)

The client never changed. No new auth code, no new library. Just a different string.

## 5. Semantic auto-routing — `model="auto"`

Pass `"auto"` and the engine's pre-trained router picks the best model for this specific prompt, based on learned per-cluster error profiles and cost weights.

In [ ]:
# Semantic auto-routing is a separate engine capability — it requires the
# routing weights pack to be loaded. If they're not, skip this cell with
# a clear message instead of letting the OpenAI SDK raise BadRequestError.
if not ROUTING_AVAILABLE:
    print("skipped: engine has no routing weights loaded (num_clusters=0).")
    print("run `opentracy download weights-mmlu-v1` on the engine host, then re-run this cell.")
else:
    resp = client.with_raw_response.chat.completions.create(
        model="auto",
        messages=[{"role": "user", "content": "Prove the square root of 2 is irrational."}],
    )
    print("picked: ", resp.headers.get("x-opentracy-selected-model"))
    print("cluster:", resp.headers.get("x-opentracy-cluster-id"))
    print(resp.parse().choices[0].message.content[:400], "...")


Hard math prompt → a strong model. Easy trivia → a cheap small one. You didn't write any rules.

## 6. Streaming — unchanged

Everything streams normally, across providers. The engine translates Anthropic / Bedrock event-streams into OpenAI SSE, so your client code doesn't need per-provider branches.

In [ ]:
# Streaming works identically — just add stream=True. The engine
# translates Anthropic / Bedrock event shapes into OpenAI deltas on the
# wire so your consumer loop never changes. Guard against the final
# zero-choice chunk that some providers emit.
stream = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    stream=True,
)

for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()


## Shut down the engine

In [ ]:
# Only stop the engine if we started it inline in this notebook —
# a shared remote engine is not ours to kill.
if engine is not None:
    engine.stop()
    print('engine stopped')
else:
    print('connected to an external engine — leaving it running')


## What you just saw

- **One-line migration**: `OpenAI(base_url=...)` is the entire adoption story
- **All 13 providers** available through the same SDK — change a string, not a dependency
- **Semantic routing** with `model="auto"` — the engine picks per prompt
- **Same streaming, same tool calls** — no code changes beyond the `base_url`

In production you'd run the engine as a long-lived service (Docker Compose), not inline in a notebook. See the [self-host guide](https://docs.opentracy.ai/guides/self-host) for that.

## Next

- **03 — Semantic routing** → go deeper into `model="auto"`, the cost_weight tradeoff, restricting candidates
- **04 — Ticket classifier (end-to-end)** → a real 50-line app showing before/after cost